In [5]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None  # replaces return_all_scores=True in newer transformers
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [6]:
test_texts = [
    "I just got some really bad news",
    "Today was the best day of my life!",
    "I can't believe they did that, I'm furious",
    "I'm not sure what's going to happen next..."
]

for text in test_texts:
    results = classifier(text)
    
    # handle both output formats
    if isinstance(results[0], dict):
        scores = results  # flat list of dicts
    else:
        scores = results[0]  # nested list
    
    results_sorted = sorted(scores, key=lambda x: x['score'], reverse=True)
    
    print(f"\nText: '{text}'")
    for emotion in results_sorted:
        bar = '█' * int(emotion['score'] * 20)
        print(f"  {emotion['label']:<10} {emotion['score']:.3f}  {bar}")


Text: 'I just got some really bad news'
  disgust    0.487  █████████
  sadness    0.314  ██████
  neutral    0.077  █
  anger      0.059  █
  fear       0.032  
  surprise   0.027  
  joy        0.004  

Text: 'Today was the best day of my life!'
  joy        0.979  ███████████████████
  surprise   0.012  
  neutral    0.003  
  anger      0.003  
  sadness    0.002  
  disgust    0.001  
  fear       0.001  

Text: 'I can't believe they did that, I'm furious'
  anger      0.975  ███████████████████
  surprise   0.010  
  fear       0.005  
  neutral    0.004  
  disgust    0.003  
  sadness    0.002  
  joy        0.001  

Text: 'I'm not sure what's going to happen next...'
  neutral    0.589  ███████████
  fear       0.179  ███
  surprise   0.087  █
  sadness    0.070  █
  disgust    0.047  
  anger      0.023  
  joy        0.006  


In [7]:
my_text = "No man is an island, Entire of itself, Every man is a piece of the continent, A part of the main. If a clod be washed away by the sea, Europe is the less. As well as if a promontory were. As well as if a manor of thy friends Or of thine own were: Any mans death diminishes me, Because I am involved in mankind, And therefore never send to know for whom the bell tolls; It tolls for thee."

results = classifier(my_text)

# handle both output formats
if isinstance(results[0], dict):
    scores = results
else:
    scores = results[0]

results_sorted = sorted(scores, key=lambda x: x['score'], reverse=True)

print(f"Text: '{my_text}'\n")
for emotion in results_sorted:
    bar = '█' * int(emotion['score'] * 20)
    print(f"  {emotion['label']:<10} {emotion['score']:.3f}  {bar}")

Text: 'No man is an island, Entire of itself, Every man is a piece of the continent, A part of the main. If a clod be washed away by the sea, Europe is the less. As well as if a promontory were. As well as if a manor of thy friends Or of thine own were: Any mans death diminishes me, Because I am involved in mankind, And therefore never send to know for whom the bell tolls; It tolls for thee.'

  neutral    0.364  ███████
  disgust    0.202  ████
  sadness    0.190  ███
  fear       0.121  ██
  anger      0.079  █
  surprise   0.030  
  joy        0.012  


In [19]:
import sounddevice as sd
from scipy.io.wavfile import write
import whisper
import tempfile
import os

# recording settings
SAMPLE_RATE = 16000
DURATION = 5  # seconds to record

print(f"Recording for {DURATION} seconds... speak now!")
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='int16')
sd.wait()
print("Done recording.")

# save to a temp file
tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
write(tmp.name, SAMPLE_RATE, audio)

# transcribe
model = whisper.load_model("base")  # tiny/base/small/medium/large — base is good for dev
result = model.transcribe(tmp.name)
transcribed_text = result["text"].strip()

os.unlink(tmp.name)  # clean up temp file

print(f"\nTranscribed: '{transcribed_text}'")

Recording for 5 seconds... speak now!
Done recording.


/Users/kippbatchelor/Documents/UNI/uni_y3/GP/nlp_emotion_art/venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcribed: 'Fuck my fucking chungress life. Fuck everything to fuck off.'


In [20]:
if transcribed_text:
    results = classifier(transcribed_text)
    
    if isinstance(results[0], dict):
        scores = results
    else:
        scores = results[0]
    
    results_sorted = sorted(scores, key=lambda x: x['score'], reverse=True)
    
    print(f"Text: '{transcribed_text}'\n")
    for emotion in results_sorted:
        bar = '█' * int(emotion['score'] * 20)
        print(f"  {emotion['label']:<10} {emotion['score']:.3f}  {bar}")
else:
    print("No speech detected.")

Text: 'Fuck my fucking chungress life. Fuck everything to fuck off.'

  anger      0.845  ████████████████
  disgust    0.134  ██
  neutral    0.007  
  sadness    0.006  
  fear       0.003  
  surprise   0.003  
  joy        0.002  
